# Loading the models

In [1]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error


TUNER_RESULTS_FOLDER = "../../data/tuner_results"
model_with_t = joblib.load(f"{TUNER_RESULTS_FOLDER}/with/refit_best_model.mod")
model_wout_t = joblib.load(f"{TUNER_RESULTS_FOLDER}/wout/refit_best_model.mod")

# Loading the catalogue data

In [7]:
CATALOG_PATH = "../../data/catalogue_data_190325.csv"

catalog = pd.read_csv(CATALOG_PATH)
catalog.dropna(how="all", axis=0, inplace=True)
catalog

# Filter the data and apply the models

In [ ]:
results = catalog.copy()

def lognnan(x):
    """Get the log10 of an array, returning NaN for non-finite values."""
    return np.where(np.isfinite(x), np.log10(x), np.nan)

results["logP_true"] = lognnan(catalog["period(s)"])
results["logPdot_true"] = lognnan(catalog["pdot(1e-11s/s)"] * 1.0e-11)
results["logL_true"] = lognnan(catalog["luminosity_bol_qui(e33erg/s)"] * 1.0e33)
results["logt_true"] = lognnan(catalog["tau_real(kyr)"] * 1.0e3)
results["logB_true"] = lognnan(catalog["Bp(e14G)"] * 1.0e14)

X = results.rename(
    columns={col: col.split("_true")[0] for col in results.columns})

X_scaled = model_wout_t.transformer.transform("X", X)
in_bounds = (X_scaled.min(axis=1) >= 0) & (X_scaled.max(axis=1) <= 1)
print(f"{sum(in_bounds)} sources with defined inputs, and in bounds "
      f"[{100*np.mean(in_bounds):.1f}%].")
results["in_bounds"] = in_bounds

X_scaled2 = model_with_t.transformer.transform("X", X)
in_bounds2 = (X_scaled2.min(axis=1) >= 0) & (X_scaled2.max(axis=1) <= 1)

assert not (~in_bounds & in_bounds2).any()

Y_reg_wout, Y_cls_wout = model_wout_t.predict(X)
Y_reg_wout.index = results.index
Y_cls_wout.index = results.index

Y_reg_with, Y_cls_with = model_with_t.predict(X)
Y_reg_with.index = results.index
Y_cls_with.index = results.index

Y_reg_with[~in_bounds2] = np.nan
Y_reg_wout[~in_bounds] = np.nan
Y_cls_with[~in_bounds2] = np.nan
Y_cls_wout[~in_bounds] = np.nan

wout_cols = {col: col + "_pred_wout"for col in Y_reg_wout.columns}
with_cols = {col: col + "_pred_with" for col in Y_reg_with.columns}
results = pd.concat([results, Y_reg_wout.rename(columns=wout_cols)],
                    axis=1, ignore_index=False)
results["envelope_wout"] = Y_cls_wout
results = pd.concat([results, Y_reg_with.rename(columns=with_cols)],
                    axis=1, ignore_index=False)
results["envelope_with"] = Y_cls_with

results.to_csv("../../results/catalog_results.csv", index_label="index")
results

In [ ]:

for obs_col in results.columns:
    if not obs_col.endswith("_true"):
        continue
    observed = results[obs_col]

    col = obs_col.split("_true")[0]
    col_wout, col_with  = [col + suffix for suffix in ["_pred_wout", "_pred_with"]]
    if col_with not in results.columns and col_wout not in results.columns:
        continue

    plt.figure()
    plt.title(col)
    plt.xlabel("Observed value")
    plt.ylabel("Predicted value")
    for pred_col, model_label, color in zip([col_wout, col_with],
                                            ["Model wout t", "Model with t"],
                                            ["b", "m"]):
        if pred_col in results.columns:
            predicted = results[pred_col]

            ok = np.isfinite(observed + predicted)
            r2 = r2_score(observed[ok], predicted[ok])
            mae = mean_absolute_error(observed[ok], predicted[ok])
            label = f"{model_label} [$R^2$={r2:.2f} | MAE={mae:.2f}]"
            plt.plot(observed[ok], predicted[ok],
                     "o" if color=="b" else ".", color=color, label=label)
    plt.legend(loc="best")
    plt.show()

In [ ]:

plt.figure()
logt_c = np.log10(results["tau_c(kyr)"] * 1000)
logt = results["logt_true"].copy()
logt_pred = results["logt_pred_wout"].copy()
ok = np.isfinite(logt + logt_pred + logt_c)
logt, logt_pred, logt_c = logt[ok], logt_pred[ok], logt_c[ok]

def mae_str(x, y):
    """Get the mean absolute error string for two arrays."""
    maerr = np.mean(np.abs(x - y))
    rmse = np.sqrt(np.mean((x-y)**2.0))
    mr = r2_score(x, y)
    return f" [MAE={maerr:.2f}, RMSE={rmse:.2f}, Rsq={mr:.2f}]"

plt.plot(logt_c, logt, "o", color="C3", alpha=0.7,
         label="real vs char." + mae_str(logt_c, logt))
plt.plot(logt_c, logt_pred, "o", color="C6", alpha=0.7,
         label="pred. vs char." + mae_str(logt_c, logt_pred))
plt.plot(logt, logt_pred, "o", color="C8", alpha=0.7,
         label="pred. vs true" + mae_str(logt, logt_pred))
plt.legend(loc="upper left")
plt.xlabel("logt[yr]")
plt.ylabel("logt[yr]")
plt.show()